# Artificial Neural Networks and Backpropagation

*Pure Python implementation from scratch — no external ML libraries.*

---

## What is an Artificial Neural Network?

An **Artificial Neural Network (ANN)** is a computational model inspired by the way biological neural networks in the brain process information. It consists of:

- **Neurons (nodes)**: Processing units that receive inputs, apply weights, sum them, add a bias, and pass the result through an **activation function**.
- **Layers**:
  - **Input layer**: Receives the raw feature values.
  - **Hidden layer(s)**: Intermediate layers that learn internal representations.
  - **Output layer**: Produces the final prediction.
- **Weights**: Parameters that scale each input — learned during training.
- **Biases**: Additional parameters that shift the activation — also learned.

### Why Neural Networks?

Neural networks can approximate **any continuous function** (Universal Approximation Theorem), making them extremely flexible. They excel at:
- Non-linear classification and regression
- Pattern recognition (images, text, speech)
- Tasks where feature engineering is difficult

### Network Architecture (this notebook)

We'll build a simple **feedforward neural network** with:
- 1 hidden layer
- Sigmoid activation functions
- Trained via **backpropagation** with gradient descent

In [ ]:
from IPython.display import Image
from math import sqrt

Image(filename='1.1_neural_network.png')

## Activation Functions

An **activation function** introduces non-linearity into the network. Without it, stacking layers would be equivalent to a single linear transformation — no matter how many layers you add.

The term "rectifier" comes from the biological analogy: just as biological neurons either fire or don't (a thresholding mechanism), activation functions decide how much signal passes through.

Common activation functions: [Wikipedia reference](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)#Gaussian_Error_Linear_Unit_(GELU))

In [ ]:
from IPython.display import Image
Image(filename='1.2_neural_network.png')

### Mathematical Definitions

**Softplus** (smooth approximation of ReLU):

$$f(x) = \ln(1 + e^{x})$$

**Sigmoid** (squashes output to [0, 1]):

$$f(x) = \frac{e^{x}}{1 + e^{x}} = \frac{1}{1 + e^{-x}}$$

**ReLU** (Rectified Linear Unit — most popular in deep learning):

$$f(x) = \max(x, 0)$$

In [ ]:
from IPython.display import Image
Image(filename='1.3_neural_network.png')

### Quick Numerical Examples

In [ ]:
import math

softplus = math.log(1 + math.exp(1.29))
print('softplus(1.29) =', softplus)

In [ ]:
sigmoid = math.exp(2.14) / (1 + math.exp(2.14))
print('sigmoid(2.14) =', sigmoid)

In [ ]:
ReLU = max(2.14, 0)
print('ReLU(2.14) =', ReLU)

### Manual Forward Pass Arithmetic

Let's trace through a small example by hand (values from the diagram):

In [ ]:
# Weighted inputs for a sample neuron
print('0.24 × (-1.30) =', 0.24 * (-1.30))
print('1.53 × 2.28   =', 1.53 * 2.28)
print('3.48 + (-2.93) - 0.58 =', 3.48 + (-2.93) - 0.58)

In [ ]:
from IPython.display import Image
Image(filename='1.4_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.5_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.6_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.7_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.8_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.9_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.10_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.11_neural_network.png')

## Building the Neural Network

### Step 1: Initialize the Network

We represent the network as a list of layers, where each layer is a list of neurons. Each neuron is a dictionary with a `'weights'` key containing random initial values.

- Hidden layer: each neuron has `n_inputs + 1` weights (the +1 is the **bias**).
- Output layer: each neuron has `n_hidden + 1` weights.

In [ ]:
from random import seed
from random import random


def initialize_our_neural_network(n_inputs, n_hidden, n_outputs):
    neural_network = list()
    hidden_layer = [{'weights': [random() for i in range(n_inputs + 1)]} for i in range(n_hidden)]
    neural_network.append(hidden_layer)
    output_layer = [{'weights': [random() for i in range(n_hidden + 1)]} for i in range(n_outputs)]
    neural_network.append(output_layer)
    return neural_network

seed(1)
network = initialize_our_neural_network(2, 1, 2)

for layer in network:
    print(layer)

In [ ]:
from IPython.display import Image
Image(filename='1.12_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.13_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.14_neural_network.png')

## Step 2: Forward Propagation

Forward propagation computes the output of the network for a given input by passing data through each layer sequentially.

Three sub-steps:
1. **Neuron activation**: Compute the weighted sum of inputs plus bias.
2. **Neuron transfer**: Apply the activation function (sigmoid) to the weighted sum.
3. **Forward propagate**: Feed each layer's outputs as the next layer's inputs.

$$\text{activation} = \text{bias} + \sum_{i=1}^{n} w_i \cdot x_i$$

In [ ]:
def neuron_activation(weights, inputs):
    activation = weights[-1]  # bias is stored as the last weight
    for i in range(len(weights) - 1):
        activation += weights[i] * inputs[i]
    return activation

In [ ]:
from math import exp

def neuron_transfer(activation):
    """Sigmoid transfer function: squash to (0, 1)."""
    result = 1.0 / (1.0 + exp(-activation))
    return result

In [ ]:
def forward_propagate(network, row):
    inputs = row
    for layer in network:
        new_inputs = []
        for neuron in layer:
            activation = neuron_activation(neuron['weights'], inputs)
            neuron['output'] = neuron_transfer(activation)
            new_inputs.append(neuron['output'])
        inputs = new_inputs
    return inputs

### Test Forward Propagation

Let's verify with a small hand-crafted network:

In [ ]:
network = [[{'weights': [0.13436424411240122,
                       0.8474337369372327,
                       0.763774618976614]}],
         [{'weights': [0.2550690257394217,
                       0.49543508709194095]},
          {'weights': [0.4494910647887381,
                       0.651592972722763]}]]

row = [1, 0]
output = forward_propagate(network, row)
print(output)

## Step 3: Backpropagation

**Backpropagation** is the algorithm that computes gradients of the loss function with respect to each weight, allowing us to update weights via gradient descent.

### The Chain Rule

The key insight is the **chain rule of calculus**. For a network with loss L, output o, and weight w:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial o} \cdot \frac{\partial o}{\partial \text{net}} \cdot \frac{\partial \text{net}}{\partial w}$$

where `net` is the weighted sum before the activation function.

### Loss Functions

| Loss | Formula | Used for |
|---|---|---|
| **MSE** | $L = \frac{1}{n}\sum(y - \hat{y})^2$ | Regression |
| **Cross-entropy** | $L = -\sum y \log(\hat{y})$ | Classification |

In this notebook we use the **sum of squared errors** for simplicity.

### Sigmoid Derivative

For the sigmoid function $\sigma(x) = \frac{1}{1+e^{-x}}$, its derivative has the elegant form:

$$\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))$$

This makes computation very efficient — we can compute the derivative directly from the output.

In [ ]:
from IPython.display import Image
Image(filename='1.15_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.16_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.17_neural_network.png')

In [ ]:
from IPython.display import Image
Image(filename='1.18_neural_network.png')

In [ ]:
def neuron_transfer_derivative(output):
    """Derivative of sigmoid, computed from the output itself."""
    derivative = output * (1 - output)
    return derivative

### Error Signal Computation

**Output layer** error:

$$\text{error} = (\text{expected} - \text{output}) \times \sigma'(\text{output})$$

**Hidden layer** error (propagated backward):

$$\text{error} = \left(\sum_k w_k \cdot \delta_j\right) \times \sigma'(\text{output})$$

where $w_k$ is the weight connecting the hidden neuron to the $k$-th output neuron, and $\delta_j$ is the error signal (delta) from the $j$-th output neuron.

In [ ]:
def backward_propagate_error(network, expected):
    for i in reversed(range(len(network))):
        layer = network[i]
        errors = list()
        if i != len(network) - 1:
            # Hidden layer: error is weighted sum of next layer's deltas
            for j in range(len(layer)):
                error = 0.0
                for neuron in network[i + 1]:
                    error += (neuron['weights'][j] * neuron['delta'])
                errors.append(error)
        else:
            # Output layer: error = expected - actual output
            for j in range(len(layer)):
                neuron = layer[j]
                errors.append(expected[j] - neuron['output'])
        for j in range(len(layer)):
            neuron = layer[j]
            neuron['delta'] = errors[j] * neuron_transfer_derivative(neuron['output'])

### Test Backpropagation

In [ ]:
network = [[{'output': 0.71, 'weights': [0.13436424411240122,
                       0.8474337369372327,
                       0.763774618976614]}],
         [{'output': 0.62, 'weights': [0.2550690257394217,
                       0.49543508709194095]},
          {'output': 0.65, 'weights': [0.4494910647887381,
                       0.651592972722763]}]]

expected = [0, 1]
backward_propagate_error(network, expected)

for layer in network:
    print(layer)

## Step 4: Update Weights

After computing deltas via backpropagation, we update each weight using **stochastic gradient descent**:

$$w = w + \alpha \cdot \delta \cdot x$$

where $\alpha$ is the learning rate, $\delta$ is the neuron's error signal, and $x$ is the input to that weight.

For hidden layers, the "input" is the output from the previous layer (not the original features).

In [ ]:
def update_weights(network, row, learning_rate):
    for i in range(len(network)):
        inputs = row[:-1]
        if i != 0:
            # For non-input layers, use the previous layer's outputs
            inputs = [neuron['output'] for neuron in network[i - 1]]
        for neuron in network[i]:
            for j in range(len(inputs)):
                neuron['weights'][j] += learning_rate * neuron['delta'] * inputs[j]
            neuron['weights'][-1] += learning_rate * neuron['delta']  # bias update

## Supplementary: Encoding Categorical Data

When preparing classification targets for a neural network, we typically use one of two encoding schemes:

### 1. Integer Encoding

Assign each category a unique integer:

| Blue | Red | Green | Yellow |
|------|-----|-------|--------|
| 1 | 2 | 3 | 4 |

**Problem**: The model may incorrectly interpret ordinal relationships (e.g., Yellow > Green).

### 2. One-Hot Encoding (preferred for neural networks)

Represent each category as a binary vector:

| | Blue | Red | Green | Yellow |
|---|------|-----|-------|--------|
| Blue | 1 | 0 | 0 | 0 |
| Red | 0 | 1 | 0 | 0 |
| Green | 0 | 0 | 1 | 0 |
| Yellow | 0 | 0 | 0 | 1 |

This avoids the ordinal assumption and works well with neural network output layers (one output neuron per class).

## Step 5: Train the Full Network

The training loop combines all previous steps:
1. **Forward propagate** to compute outputs.
2. Create a **one-hot encoded** expected vector.
3. Compute the **sum of squared errors** for monitoring.
4. **Backpropagate** errors through the network.
5. **Update weights** via gradient descent.
6. Repeat for all training rows, for all epochs.

In [ ]:
def train_our_network(network, train, learning_rate, n_epoch, n_output):
    for epoch in range(n_epoch):
        sum_error = 0
        for row in train:
            outputs = forward_propagate(network, row)
            expected = [0 for i in range(n_output)]
            expected[row[-1]] = 1
            sum_error += sum([(expected[i] - outputs[i]) ** 2 for i in range(len(expected))])
            backward_propagate_error(network, expected)
            update_weights(network, row, learning_rate)
        print("Epoch [%d], Learning rate: [%.3f], Error: [%.3f]"
              % (epoch, learning_rate, sum_error))

In [ ]:
seed(1)

dataset = [
    [2.1, 2.8, 0],
    [1.3, 2.7, 0],
    [1.2, 5.2, 0],
    [3.3, 2.8, 0],
    [1.2, 1.1, 0],
    [6.2, 5.8, 1],
    [8.3, 3.7, 1],
    [6.2, 2.7, 1],
    [7.3, 3.4, 1],
    [9.2, 2.1, 1]
]

n_inputs = len(dataset[0]) - 1
n_outputs = len(set(row[-1] for row in dataset))
network = initialize_our_neural_network(n_inputs, 2, n_outputs)
train_our_network(network, dataset, 0.1, 1000, n_outputs)

for layer in network:
    print(layer)

## Step 6: Make Predictions

At prediction time, we forward-propagate the input and return the index of the output neuron with the highest value.

In [ ]:
def predict(network, row):
    outputs = forward_propagate(network, row)
    return outputs.index(max(outputs))

# Use the pre-trained weights from our training run
network = [[{'weights': [-1.1866404384956928, 0.3006679439138231, 3.9117172696404685]},
            {'weights': [1.2235819285217509, -0.3158686384608762, -4.03117701360861]}],
           [{'weights': [3.933774188003338, -3.959801574023486, 0.4499036554334044]},
            {'weights': [-3.678895659767694, 4.256650475104409, -0.7298649815974946]}]]

for row in dataset:
    prediction = predict(network, row)
    print("Expected class: [%d], Predicted class: [%d]" % (row[-1], prediction))

---

## Summary

### What We Built

A complete feedforward neural network with:
- Random weight initialization
- Sigmoid activation
- Forward propagation
- Backpropagation (error computation + weight update)
- One-hot encoded targets

### Key Equations

| Step | Equation |
|---|---|
| Activation | $a = b + \sum w_i x_i$ |
| Sigmoid transfer | $\sigma(a) = \frac{1}{1 + e^{-a}}$ |
| Sigmoid derivative | $\sigma'(a) = \sigma(a)(1 - \sigma(a))$ |
| Output error | $\delta = (y - \hat{y}) \cdot \sigma'(\hat{y})$ |
| Hidden error | $\delta = (\sum w_k \delta_k) \cdot \sigma'(o)$ |
| Weight update | $w = w + \alpha \cdot \delta \cdot x$ |

### Limitations of This Implementation

- Only one hidden layer (deep networks have many).
- Only sigmoid activation (modern networks use ReLU, GELU, etc.).
- No mini-batch training, momentum, or adaptive learning rates.
- No regularization (dropout, L2, etc.).

### Next Steps
- Apply this network to the wheat seeds dataset (see `case07_wheat_classification_using_artificial_neural_network.py`).
- Experiment with different numbers of hidden neurons.
- Try different learning rates and observe convergence behavior.